# The NumPy + SymPy Toolkit

| Difficulty | ██░░░░░░░░ 2/10 |
| :--- | :--- |
| Prerequisites | 00.01–00.03 |
| Time | ~50 min |
| Colab GPU | Not needed |

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NiravRVaghasiya/maths-for-ai/blob/main/00_Prerequisites/04_numpy_sympy_toolkit.ipynb)

---

## 🎯 Learning Objective

Get hands-on with the two libraries used throughout this repository: NumPy for numerical arrays and broadcasting, and SymPy for symbolic math and plotting.

---

## 📐 Theory

This repository leans on two libraries constantly:

- **NumPy** — fast numerical arrays and vectorized operations. Every neural network, under
  the hood, is NumPy-style array arithmetic (PyTorch tensors follow the same rules).
- **SymPy** — symbolic mathematics. It manipulates expressions like $x^2 + 2x$ *exactly*,
  the way you would by hand, rather than approximating them numerically.

### NumPy arrays and shape

A NumPy array has a **shape** — a tuple describing its size along each axis. A vector of
length 5 has shape `(5,)`; a $3\times 4$ matrix has shape `(3, 4)`; a batch of 32 RGB images
of size $28\times 28$ has shape `(32, 3, 28, 28)`. Almost every bug in ML code is a shape
mismatch, so building the habit of checking `.shape` early pays off immediately.

### Broadcasting

**Broadcasting** is the rule NumPy uses to combine arrays of *different* shapes without
writing explicit loops. Two dimensions are compatible if they are equal, or if one of them
is 1 (in which case it's virtually "stretched" to match). Formally, shapes are compared from
the *right*, and missing leading dimensions are treated as 1:

$$\text{shape } (3, 4) \quad\text{and}\quad \text{shape } (4,) \quad \longrightarrow \quad \text{broadcast to } (3, 4)$$

This is exactly how a bias vector of shape `(d,)` gets added to every row of a batch of
activations with shape `(batch, d)` without writing a loop over the batch.

### SymPy basics

SymPy lets you declare symbols and manipulate expressions exactly:

```python
import sympy as sp
x = sp.symbols('x')
expr = x**2 + 2*x + 1
sp.factor(expr)     # -> (x + 1)**2
sp.diff(expr, x)     # -> 2*x + 2
```

We'll use SymPy throughout the calculus and probability modules to double-check derivatives
and integrals we compute by hand.

---

In [ ]:
# Setup
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')
import sympy as sp

## 👁️ Visual Intuition

Broadcasting is easiest to understand as "stretching" a smaller array to line up with a
bigger one. The diagram below shows a $(3,4)$ matrix combined with a $(4,)$ row vector and a
$(3,1)$ column vector.

In [ ]:
# Visualize broadcasting as shape-stretching
import numpy as np
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(13, 3.5))

M = np.ones((3, 4))
row = np.arange(4).reshape(1, 4)          # shape (1, 4) -> broadcasts across rows
col = np.arange(3).reshape(3, 1)          # shape (3, 1) -> broadcasts across columns

for ax, (title, arr) in zip(axes, [
    ("Matrix M, shape (3,4)", M),
    ("Row vector, shape (1,4)\nbroadcasts down each row", np.broadcast_to(row, (3, 4))),
    ("Column vector, shape (3,1)\nbroadcasts across each column", np.broadcast_to(col, (3, 4))),
]):
    im = ax.imshow(arr, cmap="Blues", vmin=0, vmax=4)
    ax.set_title(title, fontsize=9)
    for (i, j), val in np.ndenumerate(arr):
        ax.text(j, i, f"{val:.0f}", ha="center", va="center", fontsize=10)
    ax.set_xticks(range(4)); ax.set_yticks(range(3))

plt.tight_layout()
plt.show()

print("M + row  ->", (M + row).shape, " (row is stretched to every one of the 3 rows)")
print("M + col  ->", (M + col).shape, " (col is stretched across every one of the 4 columns)")

## 🐍 Implementation from Scratch

A hands-on tour: vectorized operations vs. explicit loops (and why the speed difference
matters at scale), plus SymPy verifying a derivative symbolically.

In [ ]:
# Vectorized NumPy vs. explicit Python loops -- same result, very different speed
import numpy as np
import time

rng = np.random.default_rng(0)
a = rng.standard_normal(200_000)
b = rng.standard_normal(200_000)

# Explicit loop version
t0 = time.perf_counter()
loop_result = np.empty_like(a)
for i in range(len(a)):
    loop_result[i] = a[i] * b[i] + 1.0
t_loop = time.perf_counter() - t0

# Vectorized version
t0 = time.perf_counter()
vec_result = a * b + 1.0
t_vec = time.perf_counter() - t0

print(f"Loop version:       {t_loop*1000:.1f} ms")
print(f"Vectorized version: {t_vec*1000:.2f} ms")
print(f"Speedup: ~{t_loop / t_vec:.0f}x")
print("Results match:", np.allclose(loop_result, vec_result))

print("\n--- SymPy: exact symbolic differentiation ---")
x = sp.symbols('x')
expr = x**3 + 2*x**2 + x
derivative = sp.diff(expr, x)
print("f(x)  =", expr)
print("f'(x) =", derivative)
print("f'(2) =", derivative.subs(x, 2), " (exact, not a floating-point approximation)")

## 🤖 Why This Matters for AI

Every notebook after this one uses NumPy's broadcasting rules to implement math without
writing manual loops — and the *identical* rules govern how PyTorch/TensorFlow tensors
combine during a forward pass. When a dense layer computes $y = Wx + b$ for a whole batch at
once, `b` (shape `(d_out,)`) is broadcast across the batch dimension of `Wx` (shape
`(batch, d_out)`) automatically. Below, we implement a tiny batched dense layer using nothing
but broadcasting, foreshadowing the matrix operations in Module 01.

In [ ]:
# A batched dense layer, y = Wx + b, relying entirely on broadcasting
import numpy as np

rng = np.random.default_rng(42)

batch_size, d_in, d_out = 5, 4, 3
X = rng.standard_normal((batch_size, d_in))   # a batch of 5 inputs, each 4-dimensional
W = rng.standard_normal((d_in, d_out)) * 0.5
b = rng.standard_normal(d_out) * 0.1          # shape (d_out,) -- a SINGLE bias vector

Y = X @ W + b   # (batch, d_in) @ (d_in, d_out) = (batch, d_out); then broadcast-add b
print("X shape:", X.shape, " W shape:", W.shape, " b shape:", b.shape)
print("Y shape:", Y.shape, " <- same bias vector `b` was broadcast to all", batch_size, "rows")

# Prove broadcasting is doing exactly what an explicit loop would do
Y_loop = np.stack([X[i] @ W + b for i in range(batch_size)])
print("Broadcasted result matches explicit per-row loop:", np.allclose(Y, Y_loop))

## 🏋️ Exercises

Six levels, in order: mechanical computation → conceptual understanding → derivation →
implementation → application → open-ended challenge. Solutions/outlines are in the collapsed
`Solution` blocks below each exercise — try before revealing.

### 🔢 COMPUTE
1. Create a NumPy array of shape `(4, 1)` and one of shape `(1, 5)`. Predict the shape of their
   sum by applying the broadcasting rule from the Theory section (compare shapes from the
   right; a dimension of size 1 stretches to match), then add them and check `.shape` against
   your prediction.

<details>
<summary>💡 Solution</summary>

Comparing from the right: the last dimensions are `1` and `5` — since one is `1`, it stretches
to `5`. The second-to-last dimensions are `4` and `1` — since one is `1`, it stretches to `4`.
Predicted shape: `(4, 5)`. Running `np.zeros((4,1)) + np.zeros((1,5))` gives `.shape == (4, 5)`,
matching the prediction — this is the dimensional-reasoning step broadcasting always needs
before trusting a result.

</details>

### 💭 UNDERSTAND
2. Two arrays of shape `(3, 4)` and `(3, 5)` **cannot** be broadcast together (NumPy raises a
   `ValueError`). Using the broadcasting rule (compare from the right; dimensions are compatible
   only if equal or one of them is 1), explain in one sentence exactly which pair of dimensions
   causes the incompatibility.

<details>
<summary>💡 Solution</summary>

Comparing from the right, the last dimensions are `4` and `5` — neither equals the other, and
neither is `1`, so this pair fails the compatibility rule immediately (the second-to-last
dimensions, both `3`, are never even reached as a potential problem, since the rightmost
mismatch already blocks broadcasting). If either the `4` or the `5` had instead been `1`, the
shapes would broadcast successfully.

</details>

### ✏️ DERIVE
3. Use SymPy to symbolically compute the derivative of $f(x) = \sin(x^2)$ and simplify it to a
   single expression. Then, working by hand using the chain rule (recall: you'll formalize this
   fully in `02.02`, but the mechanical rule $\frac{d}{dx}g(h(x)) = g'(h(x))\cdot h'(x)$ is
   usable now), derive the same derivative yourself with $g(u)=\sin(u)$, $h(x)=x^2$, and confirm
   your by-hand result matches SymPy's.

<details>
<summary>💡 Solution outline</summary>

With $g(u) = \sin(u)$ and $h(x) = x^2$: $g'(u) = \cos(u)$ and $h'(x) = 2x$. The chain rule
gives $\frac{d}{dx}\sin(x^2) = g'(h(x))\cdot h'(x) = \cos(x^2)\cdot 2x = 2x\cos(x^2)$.
`sp.diff(sp.sin(x**2), x)` returns `2*x*cos(x**2)` — the identical expression, confirming the
by-hand chain-rule application.

</details>

### 🐍 IMPLEMENT
4. Numerically evaluate both $f(x)=\sin(x^2)$ and your derived derivative $2x\cos(x^2)$ at
   $x=1$ using plain NumPy (`np.sin`, `np.cos`), and confirm they match SymPy's exact
   (`.subs(x, 1)`) value to at least 6 decimal places.

<details>
<summary>✅ How to know you're right</summary>

`np.sin(1.0**2)` should match `float(sp.sin(x**2).subs(x, 1))` to full float64 precision (both
are computing the same mathematical quantity, just via different code paths — SymPy
numerically, NumPy directly). Likewise `2 * 1.0 * np.cos(1.0**2)` should match
`float(sp.diff(sp.sin(x**2), x).subs(x, 1))`. Both comparisons should agree to at least 6, and
in practice 14–15, decimal places.

</details>

### 🤖 APPLY
5. A dense layer's forward pass $Y = XW + b$ relies on broadcasting to add the same bias vector
   `b` (shape `(d_out,)`) to every row of `XW` (shape `(batch, d_out)`). Now suppose instead you
   have a **per-sample** bias — a different bias vector for every item in the batch, shape
   `(batch, d_out)` — being added to a **shared** per-feature scale, shape `(d_out,)`, computed
   as `X_scaled = X * scale + per_sample_bias`. Work out (using the broadcasting rule) what
   shape `scale` needs to be for this to broadcast correctly against an `X` of shape
   `(batch, d_in)`, implement it, and confirm the result matches an explicit loop over the batch
   dimension.

<details>
<summary>✅ What you should observe</summary>

For `X` of shape `(batch, d_in)`, a per-feature `scale` needs shape `(d_in,)` (or equivalently
`(1, d_in)`) so it broadcasts across the batch dimension the same way a bias does — comparing
from the right, `d_in` matches `d_in`, and the missing leading dimension is treated as `1` and
stretched to `batch`. The broadcasted computation should match
`np.stack([X[i] * scale + per_sample_bias[i] for i in range(batch)])` exactly (`np.allclose`
returns `True`), the same "broadcasting equals an explicit loop" check the notebook's own AI
section already used for the plain dense-layer case.

</details>

### 🚀 CHALLENGE
6. Implement a vectorized pairwise Euclidean distance function `pdist(X)` that takes an `(n,
   d)` array of `n` points in `d` dimensions and returns an `(n, n)` matrix of distances, using
   ONLY broadcasting (no loops, no `scipy.spatial`). Hint: reshape `X` to `(n, 1, d)` and `(1,
   n, d)` and let broadcasting produce an `(n, n, d)` difference tensor. Verify your result
   against `scipy.spatial.distance.cdist(X, X)` (used only for checking, not inside your
   implementation), confirm the diagonal is exactly `0` (every point's distance to itself), and
   confirm the result is symmetric. Then explain how this exact reshape-and-broadcast pattern
   reappears when computing attention scores between every pair of tokens in a sequence.

<details>
<summary>🔍 What a strong answer covers</summary>

A correct implementation computes `diff = X[:, None, :] - X[None, :, :]` (shape `(n, n, d)`,
via the `(n,1,d)`/`(1,n,d)` broadcast the hint describes), then
`np.sqrt((diff ** 2).sum(axis=-1))` (shape `(n, n)`). A strong answer explicitly checks all
three things asked for — matches `cdist` via `np.allclose`, diagonal is exactly `0.0` (not just
close to it, since `X[i]-X[i]` is exactly the zero vector in floating point), and
`np.allclose(D, D.T)` for symmetry — rather than only checking the library match and assuming
the rest. The connection to attention: computing every pairwise *distance* between $n$ points
is structurally identical to computing every pairwise *dot product* between $n$ query/key
vectors (`Q @ K.T`, recall this is previewed in `01.10`) — both are $O(n^2)$ all-pairs
computations expressed as a single broadcasted/matrix operation instead of a double loop over
sequence positions, which is exactly what makes attention computationally practical to
vectorize on a GPU.

---

## 📚 Further Reading
- [NumPy: Broadcasting rules (official docs)](https://numpy.org/doc/stable/user/basics.broadcasting.html)
- [SymPy Tutorial (official docs)](https://docs.sympy.org/latest/tutorials/intro-tutorial/index.html)

---

*Next notebook: [Vectors and Vector Spaces](../01_Linear_Algebra/01_vectors_and_spaces.ipynb)*